# Lab 03-02: Prompt Augmentation

**Module:** 03 -- Application Development (30% of exam)  
**UI Sections:** Catalog | Vector Search  
**Estimated Time:** 45--60 minutes  
**Difficulty:** Intermediate

---

## Learning Objectives

- Understand why LLMs need external context to answer domain-specific questions
- Build prompt templates that inject retrieved context into the LLM prompt
- Implement a complete Retrieve-Augment-Generate (RAG) pipeline
- Manage context window limits and avoid the "lost in the middle" problem
- Compare augmentation patterns: simple stuffing, map-reduce, and refine

---

## Exam Objectives Covered

- Prompt augmentation / RAG pattern
- Context window management
- Augmentation strategies and their trade-offs
- Difference between prompt engineering and prompt augmentation

## What & Why

**Prompt augmentation** is the practice of injecting retrieved context into the prompt
before sending it to the LLM. This is the **"A" in RAG** (Retrieve-Augment-Generate).

Without augmentation, the LLM only knows what it learned during training. It cannot answer
questions about your company's policies, your product documentation, or any data created
after its training cutoff. When asked, it will either hallucinate a plausible-sounding
but wrong answer, or admit it does not know.

With augmentation, you **paste the relevant document chunks directly into the prompt**.
The LLM reads them as part of the input and generates an answer grounded in your actual
data. No model weights change. No fine-tuning is needed. You simply give the model the
information it needs at inference time.

This is the single most important pattern on the exam for Module 03. Every RAG question
assumes you understand how retrieved context flows into the prompt.

## Mental Model

> **Prompt augmentation is like giving a student an open-book exam.**
>
> The student (LLM) is smart but does not have your company's documents memorized.
> By pasting the relevant pages (retrieved chunks) into the question (prompt), you
> turn a closed-book exam into an open-book one.
>
> The student's intelligence has not changed -- they did not study more or retrain
> their brain. They simply have reference material in front of them now. That is
> exactly what prompt augmentation does: it provides reference material at inference
> time without changing the model itself.

## Exam Alert

| # | Trap (Wrong Answer) | Correct Understanding |
|---|---|---|
| 1 | "RAG fine-tunes the model on your data" | RAG injects context at **inference time** -- no model weights change. Fine-tuning is a separate process that updates weights during training. |
| 2 | "More context = better answers" | Too much context causes the **"lost in the middle"** problem -- the model pays most attention to the beginning and end, losing information in the middle. Retrieve only the most relevant chunks. |
| 3 | "Prompt augmentation and prompt engineering are the same" | **Prompt engineering** = crafting instructions, formatting, and tone. **Prompt augmentation** = injecting retrieved data. They are complementary but distinct. |

---

## Prerequisites

- **Workspace:** Databricks workspace with Foundation Model APIs enabled
- **Catalog:** `workspace.genai_labs` schema accessible
- **Prior Labs:** Lab 02-02 (Chunking Strategies) recommended but not required
- **Navigation:** **UI -->** Left nav --> **Catalog** --> **main** --> **genai_labs**

## Setup

In [ ]:
import os
from openai import OpenAI

# --- Configuration --------------------------------------------------------
MODEL = "databricks-meta-llama-3-3-70b-instruct"
SCHEMA = "workspace.genai_labs"

# --- Client setup ---------------------------------------------------------
# Works on Databricks notebooks (auto-token) and locally (env var)
client = OpenAI(
    api_key=os.environ.get("DATABRICKS_TOKEN", os.environ.get("OPENAI_API_KEY")),
    base_url=os.environ.get(
        "DATABRICKS_BASE_URL",
        "https://YOUR_WORKSPACE.cloud.databricks.com/serving-endpoints",
    ),
)

def ask(messages, temperature=0.0, max_tokens=512):
    """Send messages to the LLM and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

print(f"Model:  {MODEL}")
print(f"Schema: {SCHEMA}")
print("Client ready.")

---

## Step 1: The Problem -- LLM Without Context

Let us start by asking the LLM a question about a fictional company's internal data.
The model has never seen this information during training. What happens?

**Expected:** The model either hallucinates a plausible-sounding but wrong answer,
or admits it does not know. This motivates why we need prompt augmentation.

In [ ]:
# Ask the LLM about fictional company data it has never seen
question = "What is Acme Corp's return policy for electronics purchased after January 2025?"

messages = [
    {"role": "system", "content": "You are a helpful customer service assistant for Acme Corp."},
    {"role": "user", "content": question},
]

response_without_context = ask(messages)

print("QUESTION:")
print(f"  {question}")
print()
print("LLM RESPONSE (without context):")
print("-" * 60)
print(response_without_context)
print("-" * 60)
print()
print("Notice: The model either hallucinated a policy or admitted it does not know.")
print("This is the core problem that prompt augmentation solves.")

---

## Step 2: Building a Context Store (Simulated Vector Search)

In production, you would use Databricks Vector Search to retrieve relevant document
chunks. For this lab, we simulate the retrieval step with a local collection of
document chunks and a simple keyword-based search function.

This lets us focus on the **augmentation** part of RAG without needing a running
Vector Search endpoint.

**UI -->** Left nav --> **Catalog** --> **main** --> **genai_labs** --> In production,
your chunks would be stored in a Delta table synced to a Vector Search index.

In [ ]:
# Simulated document chunks (what Vector Search would return in production)
KNOWLEDGE_BASE = [
    {
        "id": "policy-returns-001",
        "content": (
            "Acme Corp Return Policy (Effective January 2025): Electronics may be "
            "returned within 45 days of purchase with original receipt and packaging. "
            "Items must be in original condition with all accessories included. "
            "Opened software and digital downloads are non-refundable. A 15% "
            "restocking fee applies to items returned after 30 days but within "
            "the 45-day window."
        ),
        "source": "acme_policy_handbook_v3.pdf",
    },
    {
        "id": "policy-returns-002",
        "content": (
            "Acme Corp Refund Processing: Refunds are issued to the original payment "
            "method within 5-7 business days of receiving the returned item at our "
            "warehouse. Store credit is available as an alternative and is processed "
            "immediately upon return approval. For purchases over $500, a manager "
            "approval is required before processing the refund."
        ),
        "source": "acme_policy_handbook_v3.pdf",
    },
    {
        "id": "policy-warranty-001",
        "content": (
            "Acme Corp Warranty Coverage: All electronics carry a 2-year manufacturer "
            "warranty covering defects in materials and workmanship. The warranty does "
            "not cover accidental damage, water damage, or unauthorized modifications. "
            "Extended warranty plans (3-year and 5-year) are available for purchase "
            "within 30 days of the original purchase date."
        ),
        "source": "acme_warranty_guide.pdf",
    },
    {
        "id": "policy-shipping-001",
        "content": (
            "Acme Corp Shipping Policy: Standard shipping is free on orders over $50. "
            "Express shipping (2-day) costs $12.99 and overnight shipping costs $24.99. "
            "Items are shipped from our regional warehouses and delivery times vary "
            "by location. Tracking numbers are emailed within 24 hours of shipment."
        ),
        "source": "acme_shipping_faq.pdf",
    },
    {
        "id": "policy-returns-003",
        "content": (
            "Acme Corp Holiday Return Extension: Items purchased between November 15 "
            "and December 31 may be returned until January 31 of the following year. "
            "This extension applies to all product categories including electronics. "
            "The standard return conditions (original receipt, packaging, condition) "
            "still apply during the extended return window."
        ),
        "source": "acme_policy_handbook_v3.pdf",
    },
    {
        "id": "hr-benefits-001",
        "content": (
            "Acme Corp Employee Benefits: Full-time employees receive health insurance, "
            "dental coverage, 401(k) matching up to 6%, and 20 days of paid time off. "
            "Part-time employees working over 20 hours per week receive prorated "
            "benefits. Annual enrollment opens in November each year."
        ),
        "source": "acme_hr_handbook.pdf",
    },
    {
        "id": "product-smarttv-001",
        "content": (
            "Acme SmartTV 55-inch Model X900: Features 4K OLED display, built-in "
            "streaming apps, voice control via Acme Assistant, and HDMI 2.1 ports. "
            "Retail price $899.99. Available in matte black and silver finishes. "
            "Includes wall-mount bracket and 2-year standard warranty."
        ),
        "source": "acme_product_catalog_2025.pdf",
    },
]

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} chunks")
print()
for chunk in KNOWLEDGE_BASE:
    print(f"  [{chunk['id']:<24}] {chunk['content'][:70]}...")

In [ ]:
def retrieve(query, top_k=3):
    """
    Simple keyword-based retrieval (simulates Vector Search).
    
    In production, this would be replaced by:
        results = vector_search_index.similarity_search(query=query, num_results=top_k)
    
    For this lab, we score chunks by counting keyword overlaps.
    """
    query_terms = set(query.lower().split())
    
    scored = []
    for chunk in KNOWLEDGE_BASE:
        content_terms = set(chunk["content"].lower().split())
        # Simple relevance: count overlapping words
        overlap = len(query_terms & content_terms)
        scored.append((overlap, chunk))
    
    # Sort by relevance (descending) and return top-k
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for score, chunk in scored[:top_k]]

# Test retrieval
query = "What is Acme Corp's return policy for electronics purchased after January 2025?"
retrieved_chunks = retrieve(query, top_k=3)

print(f"Query: {query!r}")
print(f"Retrieved {len(retrieved_chunks)} chunks:")
print()
for i, chunk in enumerate(retrieved_chunks):
    print(f"  Chunk {i+1} [{chunk['id']}] from {chunk['source']}:")
    print(f"    {chunk['content'][:100]}...")
    print()

print("Notice: The most relevant chunks (return policy, refund, holiday extension)")
print("were retrieved, while unrelated chunks (HR benefits, shipping) were filtered out.")

---

## Step 3: The Augmented Prompt Template

The prompt template is the bridge between retrieval and generation. It defines:

1. **System message** -- sets the LLM's role and instructions
2. **Context block** -- the retrieved chunks, clearly delimited
3. **User question** -- the original query

The key design decisions are:
- How to delimit the context (XML tags, markdown headers, triple backticks)
- Whether to include source metadata alongside each chunk
- How to instruct the model to use the context (and what to do when it is insufficient)

In [ ]:
# --- Prompt Template -------------------------------------------------------

SYSTEM_TEMPLATE = """You are a helpful customer service assistant for Acme Corp.
Answer the user's question using ONLY the provided context documents.
If the context does not contain enough information to answer, say:
"I don't have enough information in our documents to answer that question."
Do not make up information. Cite the source document when possible."""

CONTEXT_TEMPLATE = """<context>
{context}
</context>"""

USER_TEMPLATE = """Based on the context provided above, please answer:
{question}"""


def format_context(chunks):
    """Format retrieved chunks into a single context string with source attribution."""
    formatted_parts = []
    for i, chunk in enumerate(chunks, 1):
        formatted_parts.append(
            f"[Document {i} | Source: {chunk['source']}]\n{chunk['content']}"
        )
    return "\n\n".join(formatted_parts)


def build_augmented_prompt(question, chunks):
    """Build the full message list with retrieved context injected."""
    context_str = format_context(chunks)
    
    messages = [
        {"role": "system", "content": SYSTEM_TEMPLATE},
        {
            "role": "user",
            "content": CONTEXT_TEMPLATE.format(context=context_str)
                       + "\n\n"
                       + USER_TEMPLATE.format(question=question),
        },
    ]
    return messages


# --- Show the full prompt before sending -----------------------------------
question = "What is Acme Corp's return policy for electronics purchased after January 2025?"
chunks = retrieve(question, top_k=3)
messages = build_augmented_prompt(question, chunks)

print("FULL AUGMENTED PROMPT")
print("=" * 70)
for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"])
    print("-" * 70)

# Count approximate tokens
approx_chars = sum(len(m["content"]) for m in messages)
print(f"\nApprox prompt size: {approx_chars:,} characters (~{approx_chars // 4:,} tokens)")

**Expected output:**

```
FULL AUGMENTED PROMPT
======================================================================
[SYSTEM]
You are a helpful customer service assistant for Acme Corp.
Answer the user's question using ONLY the provided context documents.
...
----------------------------------------------------------------------
[USER]
<context>
[Document 1 | Source: acme_policy_handbook_v3.pdf]
Acme Corp Return Policy (Effective January 2025): Electronics may be...

[Document 2 | Source: acme_policy_handbook_v3.pdf]
Acme Corp Holiday Return Extension: ...

[Document 3 | Source: acme_policy_handbook_v3.pdf]
Acme Corp Refund Processing: ...
</context>

Based on the context provided above, please answer:
What is Acme Corp's return policy for electronics purchased after January 2025?
----------------------------------------------------------------------
```

Notice the clear structure: system instructions, delimited context with source metadata,
and the user question. This template makes it easy for the LLM to distinguish
instructions from context from the question.

---

## Step 4: RAG = Retrieve + Augment + Generate

Now we run the full pipeline end-to-end and compare the augmented answer to the
answer we got in Step 1 (without context).

In [ ]:
def rag_pipeline(question, top_k=3):
    """Full RAG pipeline: Retrieve -> Augment -> Generate."""
    # Step A: Retrieve relevant chunks
    chunks = retrieve(question, top_k=top_k)
    
    # Step B: Augment the prompt with retrieved context
    messages = build_augmented_prompt(question, chunks)
    
    # Step C: Generate the answer
    answer = ask(messages)
    
    return {
        "question": question,
        "chunks_used": len(chunks),
        "sources": [c["source"] for c in chunks],
        "answer": answer,
    }


# --- Run the pipeline -----------------------------------------------------
question = "What is Acme Corp's return policy for electronics purchased after January 2025?"
result = rag_pipeline(question)

print("RAG PIPELINE RESULT")
print("=" * 70)
print(f"Question:     {result['question']}")
print(f"Chunks used:  {result['chunks_used']}")
print(f"Sources:      {', '.join(set(result['sources']))}")
print()
print("ANSWER (with context):")
print("-" * 70)
print(result["answer"])
print("-" * 70)

print()
print("COMPARISON -- Answer WITHOUT context (from Step 1):")
print("-" * 70)
print(response_without_context)
print("-" * 70)

print()
print("The augmented answer cites specific details (45 days, 15% restocking fee,")
print("original receipt required) that the model could not know without context.")

In [ ]:
# Test with additional questions to see the pipeline in action

test_questions = [
    "Can I return opened software to Acme Corp?",
    "What warranty options does Acme offer for electronics?",
    "What are the employee benefits at Acme Corp?",
    "What is Acme's policy on cryptocurrency payments?",  # not in knowledge base
]

for q in test_questions:
    result = rag_pipeline(q, top_k=2)
    print(f"Q: {q}")
    print(f"A: {result['answer'][:200]}...")
    print(f"   Sources: {', '.join(set(result['sources']))}")
    print()

**Key observations from Step 4:**
- The augmented answer cites specific details (45 days, 15% restocking fee) from the context
- When context is available, the model answers accurately with source citations
- When context is insufficient (cryptocurrency question), the model says so instead of hallucinating
- This is because our system prompt instructs: *"If the context does not contain enough information..."*

---

## Step 5: Context Window Management

The LLM's context window has a finite size. Stuffing too many chunks into the prompt
can cause two problems:

1. **Token overflow** -- the prompt exceeds the model's context limit and gets truncated
   or rejected
2. **Lost in the middle** -- research shows that LLMs pay the most attention to the
   beginning and end of long prompts, losing track of information in the middle

Best practice: **3-5 highly relevant chunks > 20 mediocre ones.**

In [ ]:
# --- Demonstrate context window management ---------------------------------

def estimate_tokens(text):
    """Rough token estimate: ~4 characters per token for English text."""
    return len(text) // 4

def build_prompt_with_budget(question, chunks, max_context_tokens=1000):
    """
    Build an augmented prompt that respects a token budget for context.
    Adds chunks one at a time until the budget is reached.
    """
    selected_chunks = []
    running_tokens = 0
    
    for chunk in chunks:
        chunk_tokens = estimate_tokens(chunk["content"])
        if running_tokens + chunk_tokens > max_context_tokens:
            print(f"  Budget reached. Skipping chunk [{chunk['id']}] "
                  f"({chunk_tokens} tokens would exceed {max_context_tokens} limit)")
            continue
        selected_chunks.append(chunk)
        running_tokens += chunk_tokens
        print(f"  Added chunk [{chunk['id']}] ({chunk_tokens} tokens, "
              f"running total: {running_tokens})")
    
    print(f"  Final: {len(selected_chunks)} chunks, ~{running_tokens} context tokens")
    return build_augmented_prompt(question, selected_chunks)


# --- Scenario: Generous budget (all chunks fit) ----------------------------
print("SCENARIO 1: Large budget (2000 tokens)")
print("-" * 50)
question = "Tell me everything about Acme Corp's return and warranty policies."
all_chunks = retrieve(question, top_k=5)
messages_large = build_prompt_with_budget(question, all_chunks, max_context_tokens=2000)

print()
print("SCENARIO 2: Tight budget (300 tokens)")
print("-" * 50)
messages_tight = build_prompt_with_budget(question, all_chunks, max_context_tokens=300)

print()
print("SCENARIO 3: Very tight budget (100 tokens)")
print("-" * 50)
messages_tiny = build_prompt_with_budget(question, all_chunks, max_context_tokens=100)

In [ ]:
# --- The "Lost in the Middle" Problem --------------------------------------
# Research: Liu et al. (2023) "Lost in the Middle" shows LLMs attend most to
# the first and last items in a long context, neglecting middle content.

print("THE 'LOST IN THE MIDDLE' PROBLEM")
print("=" * 70)
print()
print(f"{'Position in context':<32} {'LLM attention':<18} {'Implication'}")
print("-" * 70)
print(f"{'Beginning (first 1-2 chunks)':<32} {'HIGH':<18} {'Put most important info here'}")
print(f"{'Middle (chunks 3-8)':<32} {'LOW':<18} {'Information here may be ignored'}")
print(f"{'End (last 1-2 chunks)':<32} {'HIGH':<18} {'Put question/instructions here'}")
print()
print("BEST PRACTICES FOR THE EXAM:")
print("-" * 70)
print("1. Retrieve fewer, more relevant chunks (3-5) rather than many (20+)")
print("2. Place the most relevant chunk FIRST in the context block")
print("3. Put the user question AFTER the context (at the end of the prompt)")
print("4. Use clear delimiters (<context>...</context>) so the model can")
print("   distinguish context from instructions")
print()

# Demonstrate: retrieve too many chunks vs. focused retrieval
question = "What is the return policy?"
many_chunks = retrieve(question, top_k=7)  # all 7 chunks!
few_chunks = retrieve(question, top_k=3)   # just the best 3

print(f"All 7 chunks retrieved ({sum(estimate_tokens(c['content']) for c in many_chunks)} est. tokens):")
for c in many_chunks:
    print(f"  [{c['id']:<24}] relevant? {'return' in c['content'].lower()}")

print()
print(f"Top 3 chunks only ({sum(estimate_tokens(c['content']) for c in few_chunks)} est. tokens):")
for c in few_chunks:
    print(f"  [{c['id']:<24}] relevant? {'return' in c['content'].lower()}")

print()
print("With 7 chunks, the LLM gets irrelevant noise (HR benefits, product specs, shipping).")
print("With 3 chunks, every piece of context is about returns. Focused context = better answers.")

---

## Step 6: Augmentation Patterns for the Exam

There are three main patterns for how retrieved chunks are combined with the prompt.
The exam expects you to know when to use each one.

In [ ]:
# --- Pattern 1: Simple Stuffing --------------------------------------------
# Concatenate all retrieved chunks into a single context block.
# This is what we have been doing so far.

def simple_stuffing(question, chunks):
    """Pattern 1: Stuff all chunks into one prompt."""
    messages = build_augmented_prompt(question, chunks)
    return ask(messages)


question = "Summarize Acme Corp's return and warranty policies."
chunks = retrieve(question, top_k=4)

print("PATTERN 1: Simple Stuffing")
print("=" * 70)
print("Approach: Concatenate all chunks into one context block, send once.")
print(f"Chunks: {len(chunks)} | LLM calls: 1")
print()
answer = simple_stuffing(question, chunks)
print("Answer:")
print(answer)

In [ ]:
# --- Pattern 2: Map-Reduce ------------------------------------------------
# Step 1 (Map): Summarize each chunk independently
# Step 2 (Reduce): Combine summaries into a final answer

def map_reduce(question, chunks):
    """Pattern 2: Summarize each chunk, then combine summaries."""
    # MAP: Get a summary from each chunk independently
    summaries = []
    for i, chunk in enumerate(chunks):
        map_messages = [
            {"role": "system", "content": "Extract the key facts relevant to the question. Be concise."},
            {"role": "user", "content": (
                f"Question: {question}\n\n"
                f"Document:\n{chunk['content']}\n\n"
                "Key facts from this document:"
            )},
        ]
        summary = ask(map_messages, max_tokens=150)
        summaries.append(summary)
        print(f"  MAP chunk {i+1} [{chunk['id']}]: {summary[:80]}...")
    
    # REDUCE: Combine all summaries into a final answer
    combined = "\n\n".join(f"Summary {i+1}: {s}" for i, s in enumerate(summaries))
    reduce_messages = [
        {"role": "system", "content": "Synthesize the following summaries into a single comprehensive answer."},
        {"role": "user", "content": (
            f"Question: {question}\n\n{combined}\n\nFinal answer:"
        )},
    ]
    final = ask(reduce_messages)
    return final


print("PATTERN 2: Map-Reduce")
print("=" * 70)
print("Approach: Summarize each chunk independently, then combine summaries.")
print(f"Chunks: {len(chunks)} | LLM calls: {len(chunks)} (map) + 1 (reduce) = {len(chunks) + 1}")
print()
answer = map_reduce(question, chunks)
print()
print("Final Answer:")
print(answer)

In [ ]:
# --- Pattern 3: Refine ----------------------------------------------------
# Start with the first chunk, generate an initial answer.
# Then iterate: for each subsequent chunk, refine the answer.

def refine(question, chunks):
    """Pattern 3: Iteratively refine the answer with each chunk."""
    # Initial answer from first chunk
    messages = [
        {"role": "system", "content": "Answer the question using the provided context."},
        {"role": "user", "content": (
            f"Question: {question}\n\n"
            f"Context:\n{chunks[0]['content']}\n\n"
            "Answer:"
        )},
    ]
    current_answer = ask(messages, max_tokens=300)
    print(f"  INITIAL (from chunk 1 [{chunks[0]['id']}]):")
    print(f"    {current_answer[:100]}...")
    
    # Refine with each subsequent chunk
    for i, chunk in enumerate(chunks[1:], 2):
        refine_messages = [
            {"role": "system", "content": (
                "You have an existing answer and new context. "
                "Refine the answer by incorporating any new relevant information. "
                "If the new context is not relevant, return the existing answer unchanged."
            )},
            {"role": "user", "content": (
                f"Question: {question}\n\n"
                f"Existing answer:\n{current_answer}\n\n"
                f"New context:\n{chunk['content']}\n\n"
                "Refined answer:"
            )},
        ]
        current_answer = ask(refine_messages, max_tokens=300)
        print(f"  REFINE with chunk {i} [{chunk['id']}]:")
        print(f"    {current_answer[:100]}...")
    
    return current_answer


print("PATTERN 3: Refine")
print("=" * 70)
print("Approach: Build answer iteratively, refining with each new chunk.")
print(f"Chunks: {len(chunks)} | LLM calls: {len(chunks)} (one per chunk)")
print()
answer = refine(question, chunks)
print()
print("Final Answer:")
print(answer)

In [ ]:
# --- Pattern Comparison Table -----------------------------------------------

patterns = [
    {
        "Pattern": "Simple Stuffing",
        "LLM Calls": "1",
        "Latency": "Low",
        "Cost": "Low",
        "Best For": "Short context, few chunks (< 5)",
        "Risk": "Lost in the middle with many chunks",
    },
    {
        "Pattern": "Map-Reduce",
        "LLM Calls": "N + 1",
        "Latency": "High (parallelizable)",
        "Cost": "High",
        "Best For": "Many chunks, comprehensive summary",
        "Risk": "Map step may lose nuance; higher cost",
    },
    {
        "Pattern": "Refine",
        "LLM Calls": "N",
        "Latency": "High (sequential)",
        "Cost": "Medium-High",
        "Best For": "Detailed answer; order matters",
        "Risk": "Early info diluted; sequential = slow",
    },
]

print("AUGMENTATION PATTERN COMPARISON")
print("=" * 100)
header = f"{'Pattern':<20} {'LLM Calls':<12} {'Latency':<24} {'Cost':<12} {'Best For':<35} {'Risk'}"
print(header)
print("-" * 110)
for p in patterns:
    print(f"{p['Pattern']:<20} {p['LLM Calls']:<12} {p['Latency']:<24} "
          f"{p['Cost']:<12} {p['Best For']:<35} {p['Risk']}")

print()
print("EXAM TIP: Simple stuffing is the default RAG pattern. The exam expects")
print("you to know it. Map-reduce and refine appear in questions about handling")
print("large numbers of retrieved chunks that exceed the context window.")

---

## Exam Tips & Traps

| # | Tip | Why It Matters |
|---|---|---|
| 1 | **Prompt augmentation injects context at inference time -- it does NOT change model weights.** | The most common trap. RAG is not fine-tuning. No training happens. |
| 2 | **Simple stuffing is the default RAG pattern: retrieve chunks, concatenate them, send to LLM in one call.** | If the exam asks "how does RAG work," this is the expected answer. |
| 3 | **The "lost in the middle" problem means LLMs attend more to the beginning and end of long contexts.** | This is why "more context = better" is wrong. Focused retrieval beats bulk retrieval. |
| 4 | **Prompt engineering != prompt augmentation.** Engineering crafts instructions. Augmentation injects data. | The exam tests whether you can distinguish these two concepts. |
| 5 | **Map-reduce and refine patterns solve the problem of too many chunks for one context window.** | If asked "how to handle 50 retrieved chunks," simple stuffing is wrong -- use map-reduce. |

---

## Key Takeaways

**Core Concepts**
- Prompt augmentation = injecting retrieved context into the prompt before sending to the LLM
- This is the "A" in RAG: Retrieve chunks, **Augment** the prompt, Generate the answer
- Without augmentation, the LLM only knows its training data and will hallucinate or refuse
- A well-structured prompt template separates system instructions, context, and the question

**Databricks-Specific**
- In production, the `retrieve()` function is replaced by Databricks Vector Search
- Document chunks live in a Delta table synced to a Vector Search index
- The augmented prompt is sent to Foundation Model APIs (`databricks-meta-llama-3-3-70b-instruct`)
- **UI -->** Catalog --> Vector Search indexes are where your retrieval layer connects

**Exam Essentials**
- RAG does NOT fine-tune the model. It injects context at inference time.
- 3-5 highly relevant chunks > 20 mediocre chunks (lost in the middle problem)
- Simple stuffing = 1 LLM call. Map-reduce = N+1 calls. Refine = N calls.
- Prompt augmentation and prompt engineering are different concepts.
- Always instruct the model to say "I don't know" when context is insufficient -- this prevents hallucination.

---

**Next Lab:** [03-03: Guardrails and PII](./03_guardrails_and_pii.ipynb) -- Add safety guardrails and PII detection to your RAG pipeline.